# XGBoost Tuning (Saphondanai, wk2–3)

งานคู่ขนานกับ [`04_tuning_P.ipynb`](04_tuning_P.ipynb) ของ Puripat (parallel search ตาม TASKS.md) ใช้ข้อมูล split เดียวกัน (80/20, seed 42) เพื่อเทียบกันได้ตรง ๆ แต่ค้นคนละทางใน 3 เรื่อง:

| เรื่อง | Puripat | Saphondanai (โน้ตบุ๊กนี้) |
| :--- | :--- | :--- |
| ชุดฟีเจอร์ | base + ฟีเจอร์ใหม่ 3 ตัวที่เพิ่ม CV AUC | ทดลองตัดฟีเจอร์ตาม README 4 (Localization) และคอลัมน์ค่าจ้างที่เป็น noise |
| ตัววัดตอน tune | ROC AUC | PR-AUC (average precision) เพราะคนลาออกมีแค่ 16% PR-AUC สะท้อนความแม่นในกลุ่มที่ HR สนใจมากกว่า |
| การเทียบ | — | เทียบโมเดลของทั้งสองคนบน test ชุดเดียวกัน แล้วเสนอโมเดลสุดท้าย |

MLflow: ใช้ `src/mlflow_setup.py` ถ้าตั้ง `.env` ไว้จะ log เข้า DagsHub กลางของทีม ถ้าไม่ได้ตั้งจะใช้ `mlflow.db` ในเครื่อง

In [1]:
import sys
sys.path.append("../src")
import warnings
warnings.filterwarnings("ignore")

import mlflow, mlflow.xgboost, numpy as np, optuna, pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, f1_score, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, train_test_split
from xgboost import XGBClassifier

import mlflow_setup
from clean_pipeline import RAW_FILENAME, clean_data, load_raw_data
from feature_pipeline import SELECTED_FEATURES, add_features

optuna.logging.set_verbosity(optuna.logging.WARNING)
print("MLflow:", mlflow_setup.setup("attrition-xgboost-S-tuning"))

df = add_features(clean_data(load_raw_data(f"../data/raw/{RAW_FILENAME}")), only=SELECTED_FEATURES)
X, y = df.drop(columns="Attrition"), df["Attrition"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)  # เหมือน notebook 04 ของ Puripat
spw = (ytr == 0).sum() / (ytr == 1).sum()
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
DEFAULT = dict(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
               random_state=42, scale_pos_weight=spw, n_jobs=4)

def cv_scores(cols, params=DEFAULT):
    s = cross_validate(XGBClassifier(**params), Xtr[cols], ytr, cv=cv, scoring=["roc_auc", "average_precision"])
    return {"cv_auc": s["test_roc_auc"].mean(), "cv_pr_auc": s["test_average_precision"].mean()}

print("train:", Xtr.shape, "test:", Xte.shape, "| scale_pos_weight =", round(spw, 2))

2026/09/30 12:23:07 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/09/30 12:23:07 INFO mlflow.store.db.utils: Updating database tables


2026/09/30 12:23:08 INFO mlflow.tracking.fluent: Experiment with name 'attrition-xgboost-S-tuning' does not exist. Creating a new experiment.


MLflow: sqlite:///D:\CSI\dev ink\employee-attrition-predictor\mlflow.db
train: (1176, 50) test: (294, 50) | scale_pos_weight = 5.19


## 1. ชุดฟีเจอร์ที่ทดลอง

- P_selected: ชุดสุดท้ายของ Puripat (base + `OverTimeXDistance`, `AvgSatisfaction`, `TenureRatio`) ใช้เป็นจุดอ้างอิง
- localized: ตัด `BusinessTravel` และ `StockOptionLevel` ตาม README หัวข้อ 4: สองฟีเจอร์นี้อิงโครงสร้างบริษัทอเมริกัน (การเดินทางธุรกิจ, สิทธิ์ซื้อหุ้น) พบน้อยในบริษัท/SME ไทย โมเดลที่พึ่งสองตัวนี้มากจะ transfer มาใช้ในไทยได้ไม่ดี
- localized_lean: ตัดเพิ่ม `DailyRate`, `HourlyRate`, `MonthlyRate` ซึ่งใน IBM dataset ไม่สัมพันธ์กับ `MonthlyIncome` เลย (ดู cell ถัดไป) จึงน่าจะเป็นค่าสุ่มที่ไม่มีความหมายทางธุรกิจ และบริษัทไทยไม่มีข้อมูลแบบนี้ให้กรอก

**กติกาเลือกชุด (ตั้งไว้ก่อนดูผล):** เลือกชุดที่ CV PR-AUC สูงสุด แต่ถ้าชุด localized ตัวใดตัวหนึ่งแพ้ไม่เกิน 0.01 ให้เลือกชุด localized นั้นแทน เพราะใช้งานในไทยได้จริงกว่า

In [2]:
raw_rates = load_raw_data(f"../data/raw/{RAW_FILENAME}")[["MonthlyIncome", "DailyRate", "HourlyRate", "MonthlyRate"]]
raw_rates.corr()["MonthlyIncome"].round(3)

MonthlyIncome    1.000
DailyRate        0.008
HourlyRate      -0.016
MonthlyRate      0.035
Name: MonthlyIncome, dtype: float64

In [3]:
BASE_COLS = list(Xtr.columns)
LOCAL_DROP = ["BusinessTravel", "StockOptionLevel"]
RATE_DROP = ["DailyRate", "HourlyRate", "MonthlyRate"]
SETS = {
    "P_selected": BASE_COLS,
    "localized": [c for c in BASE_COLS if c not in LOCAL_DROP],
    "localized_lean": [c for c in BASE_COLS if c not in LOCAL_DROP + RATE_DROP],
}
set_scores = pd.DataFrame({k: cv_scores(v) for k, v in SETS.items()}).T
set_scores["n_features"] = [len(v) for v in SETS.values()]
set_scores.round(4)

,cv_auc,cv_pr_auc,n_features
P_selected,0.7990,0.5913,50
localized,0.7879,0.5555,48
localized_lean,0.7927,0.5698,45


In [4]:
best_any = set_scores["cv_pr_auc"].idxmax()
local_ok = [k for k in ("localized_lean", "localized")
            if set_scores.loc[best_any, "cv_pr_auc"] - set_scores.loc[k, "cv_pr_auc"] <= 0.01]
FEATURE_SET = local_ok[0] if local_ok else best_any
COLS = SETS[FEATURE_SET]
print("ชุดที่ดีที่สุดตาม PR-AUC:", best_any, "| เลือกใช้:", FEATURE_SET, f"({len(COLS)} ฟีเจอร์)")

ชุดที่ดีที่สุดตาม PR-AUC: P_selected | เลือกใช้: P_selected (50 ฟีเจอร์)


## 2. Tune ด้วย Optuna (40 trials, maximize CV PR-AUC บน train)

ช่วงค้นหาเดียวกับของ Puripat เพื่อให้ผลต่างมาจากชุดฟีเจอร์และตัววัดเท่านั้น

In [5]:
def objective(t):
    p = dict(DEFAULT,
        n_estimators=t.suggest_int("n_estimators", 100, 600, step=50),
        max_depth=t.suggest_int("max_depth", 2, 6),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.2, log=True),
        subsample=t.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0),
        min_child_weight=t.suggest_int("min_child_weight", 1, 10),
        reg_lambda=t.suggest_float("reg_lambda", 0.1, 10, log=True),
    )
    return cv_scores(COLS, p)["cv_pr_auc"]

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40)
print(f"best CV PR-AUC = {study.best_value:.4f} (default = {set_scores.loc[FEATURE_SET, 'cv_pr_auc']:.4f})")
study.best_params

best CV PR-AUC = 0.6314 (default = 0.5913)


{'n_estimators': 600,
 'max_depth': 2,
 'learning_rate': 0.02073469202477496,
 'subsample': 0.7400349895522362,
 'colsample_bytree': 0.5300264640253618,
 'min_child_weight': 10,
 'reg_lambda': 0.34192322013810594}

## 3. เทียบกับโมเดลของ Puripat

- P คือชุด `P_selected` + hyperparameter ที่ Puripat tune ได้ใน notebook 04 (คัดลอกค่ามาตรง ๆ เทรนใหม่ด้วย seed เดียวกัน จึงได้โมเดลเดียวกัน)
- S คือชุดที่เลือกข้างบน + hyperparameter จาก Optuna รอบนี้

วัดสองแบบ: CV บน train (เสถียรกว่า เพราะ test มีคนลาออกแค่ ~47 คน) และ test ครั้งเดียว
`recall@top20%` = ถ้า HR ดูแลได้แค่ 20% ของพนักงานที่โมเดลให้คะแนนสูงสุด จะครอบคลุมคนที่ลาออกจริงกี่ %

In [6]:
P_BEST = {"n_estimators": 500, "max_depth": 2, "learning_rate": 0.03132429398213804, "subsample": 0.664952951683747,
          "colsample_bytree": 0.5305253930212119, "min_child_weight": 10, "reg_lambda": 1.2338780559789992}
CANDIDATES = {
    "P": (SETS["P_selected"], dict(DEFAULT, **P_BEST)),
    "S": (COLS, dict(DEFAULT, **study.best_params)),
}

def recall_at_top(y_true, p, frac=0.2):
    top = np.argsort(-p)[: int(len(p) * frac)]
    return y_true.to_numpy()[top].sum() / y_true.sum()

rows, models = {}, {}
for name, (cols, params) in CANDIDATES.items():
    m = XGBClassifier(**params).fit(Xtr[cols], ytr)
    p = m.predict_proba(Xte[cols])[:, 1]
    rows[name] = {**cv_scores(cols, params), "test_auc": roc_auc_score(yte, p), "test_pr_auc": average_precision_score(yte, p),
                  "test_f1": f1_score(yte, p > 0.5), "test_brier": brier_score_loss(yte, p),
                  "test_recall@top20%": recall_at_top(yte, p), "n_features": len(cols)}
    models[name] = m
compare = pd.DataFrame(rows).T
compare.round(3)

,cv_auc,cv_pr_auc,test_auc,test_pr_auc,test_f1,test_brier,test_recall@top20%,n_features
P,0.827,0.636,0.814,0.609,0.488,0.135,0.596,50.0
S,0.824,0.631,0.807,0.592,0.496,0.142,0.574,50.0


In [7]:
# log ทั้งสองตัวเข้า MLflow; register เฉพาะของ S (ของ P ให้ Puripat register เองจาก notebook 04)
for name, (cols, params) in CANDIDATES.items():
    metrics = {k.replace("@", "_at_").replace("%", "pct"): float(v) for k, v in rows[name].items()}
    tuned = P_BEST if name == "P" else study.best_params
    with mlflow.start_run(run_name=f"xgb-{name}-compare"):
        mlflow.set_tags({"author": "Saphondanai", "candidate": name, "feature_set": "P_selected" if name == "P" else FEATURE_SET})
        mlflow.log_params({**tuned, "scale_pos_weight": round(spw, 2), "n_features": len(cols)})
        mlflow.log_metrics(metrics)
        if name == "S":
            mlflow.xgboost.log_model(models[name], name="model", registered_model_name="attrition-xgboost-S")

Successfully registered model 'attrition-xgboost-S'.
Created version '1' of model 'attrition-xgboost-S'.


## 4. ปัจจัยที่โมเดล S ใช้มากที่สุด

In [8]:
pd.Series(models["S"].feature_importances_, index=models["S"].feature_names_in_).sort_values(ascending=False).head(12).round(3)

OverTime                         0.047
JobLevel                         0.046
StockOptionLevel                 0.041
TotalWorkingYears                0.038
JobRole_Sales Executive          0.038
YearsAtCompany                   0.037
YearsWithCurrManager             0.036
OverTimeXDistance                0.033
AvgSatisfaction                  0.032
Age                              0.029
MonthlyIncome                    0.027
JobRole_Laboratory Technician    0.026
dtype: float32

## สรุป / ข้อเสนอโมเดลสุดท้าย

**1. ชุดฟีเจอร์: ยืนยันชุดของ Puripat (`SELECTED_FEATURES` ใน `src/feature_pipeline.py`)**
- ตัด `BusinessTravel` + `StockOptionLevel` (localized) ทำให้ CV PR-AUC ลด 0.036 และตัดคอลัมน์อัตราค่าจ้างเพิ่ม (localized_lean) ยังลด 0.022 เกินเกณฑ์ 0.01 ที่ตั้งไว้ก่อนดูผล จึงไม่ตัดออกจากโมเดล
- ความเสี่ยงเรื่อง transfer มาใช้ในไทยจัดการด้วย `/recalibrate` (README 6.5) และคำแนะนำใน company summary ที่แปลง `StockOptionLevel` เป็นสวัสดิการระยะยาวแบบไทยแทน
- `DailyRate`/`HourlyRate`/`MonthlyRate` ไม่สัมพันธ์กับ `MonthlyIncome` เลย (|r| < 0.04) แต่การตัดออกยังทำให้ผลแย่ลงเล็กน้อย เก็บไว้ก่อน

**2. Hyperparameter: เสนอใช้ของ Puripat (โมเดล P) เป็นโมเดลสุดท้าย**
- การ tune ด้วย PR-AUC ได้ hyperparameter ใกล้เคียงกับของ Puripat มาก (max_depth 2, min_child_weight 10, colsample ~0.53) แสดงว่าทั้งสองทางหาเจอบริเวณเดียวกัน
- โมเดล P ดีกว่า S เล็กน้อยเกือบทุกตัววัดทั้ง CV และ test (ต่างกัน ≤ 0.017 ซึ่งอยู่ในระดับ noise ของ test ที่มีคนลาออกแค่ 47 คน) ยกเว้น F1 ที่ threshold 0.5
- ถ้า HR ดูแลพนักงาน 20% ที่คะแนนสูงสุด โมเดล P ครอบคลุมคนที่ลาออกจริง ~60%

**3. ข้อสังเกต:** เทรนโมเดล P ซ้ำด้วยค่าเดียวกับ notebook 04 ได้ test AUC 0.814 (ของ Puripat 0.812) ต่างกันเล็กน้อยน่าจะมาจากเวอร์ชัน library/จำนวน thread ต่างกัน ควรเทรนตัวสุดท้ายครั้งเดียวบน MLflow กลาง (DagsHub) แล้วทุกคนโหลดตัวเดียวกัน

**ต้องทำต่อร่วมกับ Puripat:** ยืนยันข้อเสนอนี้ แล้ว register โมเดล P เป็นชื่อกลางของทีมบน DagsHub และตั้ง `MODEL_URI` ใน `.env` ของทุกคนให้ชี้ตัวเดียวกัน ระหว่างนี้ `attrition-xgboost-S` v1 ใช้แทนในเครื่องได้ (ชุดฟีเจอร์เดียวกัน)